# 04. Deduplication and Record Linkage Basics
**Engineering Data Processing with Python** | one-day course | approx. 45 minutes

> **EXERCISE NOTEBOOK.** Work top to bottom. Run every cell in order (Shift+Enter). Where you see `____` replace it with your answer. Where you see `# TODO`, write the code. If an error mentions `____` (for example `NameError: name '____' is not defined`), you have a blank left to fill. Blanks are four underscores.


In [ ]:
# ===================== SETUP: run this cell first (click it, then Shift+Enter) =====================
# It makes sure the course data is available in this Colab session and loads the tools we need.
import os, zipfile, urllib.request
from pathlib import Path

DATA_ZIP_URL = ""   # if your trainer gives you a download link for data.zip, paste it between the quotes

def get_course_data():
    if Path("data").is_dir():
        print("Data folder found:", sorted(p.name for p in Path("data").iterdir())[:20])
        return
    if not Path("data.zip").exists() and DATA_ZIP_URL:
        print("Downloading course data...")
        urllib.request.urlretrieve(DATA_ZIP_URL, "data.zip")
    zip_path = Path("data.zip")
    if not zip_path.exists():
        try:
            from google.colab import files
            print("Choose data.zip from your computer (your trainer will tell you where it is).")
            uploaded = files.upload()
            zip_path = Path(next(iter(uploaded)))
        except ImportError:
            raise FileNotFoundError("Put data.zip or the data folder next to this notebook.")
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(".")
    print("Data ready:", sorted(p.name for p in Path("data").iterdir()))

get_course_data()
Path("outputs").mkdir(exist_ok=True)

import pandas as pd
import numpy as np
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

def check(condition, ok="Looks right.", hint="Not quite yet. Re-read the instructions above the cell."):
    """Tiny self-check helper used throughout the course."""
    print(("PASS: " + ok) if condition else ("NOT YET: " + hint))

## What you will do

Duplicates inflate counts and costs. Removing them carelessly deletes real records. This module is about doing it
**safely**:

1. Decide whether duplicates are errors at all (concrete tests).
2. Remove exact duplicate rows.
3. Remove duplicate *keys* (same work order exported twice with different content) using a rule you can defend.
4. Find conflicts you cannot resolve automatically and send them to a human.
5. Match people's names that are spelt differently (fuzzy matching), with a human approving the matches.
6. Make IDs from an old system match the new one.

**Starting point:** the trainer's checkpoint of Module 03's clean outputs, so everyone starts from the same data.

In [ ]:
DATES = ["date_raised", "date_closed", "last_modified"]
wo = pd.read_csv("data/checkpoints/m03_work_orders_clean.csv", parse_dates=DATES)
wo["priority"] = wo["priority"].astype("Int64")
reg = pd.read_csv("data/checkpoints/m03_asset_register_clean.csv", parse_dates=["install_date"])
print(wo.shape, reg.shape)

## 1. Are duplicates always errors?

### Exercise 4.1 [Level 1: fill the blanks]

In [ ]:
concrete = pd.read_csv("data/concrete.csv")
dup_mask = concrete.____(keep=False)        # keep=False marks EVERY copy, not just the 2nd onwards
print("Rows that have an identical twin:", dup_mask.sum())
concrete[dup_mask].sort_values(list(concrete.columns)).head(6)

Same mix, same age, same strength to eight decimal places. Two possibilities:

* **Error**: the same test result was entered twice. Remove the copy, otherwise this mix counts double in any model.
* **Genuine**: replicate specimens. But replicate cubes almost never give strengths identical to 8 decimal places.

Here the evidence points to copy errors, so removing them is defensible. The important bit is that you **looked and
decided**, and you can say why. Write it down next to the code.

In [ ]:
concrete_dedup = concrete.____()
print(len(concrete), "->", len(concrete_dedup), "rows")

## 2. Exact duplicate work orders

### Exercise 4.2 [Level 1: fill the blanks]

In [ ]:
n_before = len(wo)
print("Exact duplicate rows:", wo.duplicated().____())
wo = wo.____()
print(n_before, "->", len(wo))

## 3. Duplicate keys: same work order, different content

Every work order should appear once. Is `wo_id` unique now?

In [ ]:
print("wo_id unique?", wo["wo_id"].is_unique)
repeat = wo[wo["wo_id"].duplicated(keep=False)].sort_values(["wo_id", "last_modified"])
print("Rows sharing a wo_id with another row:", len(repeat))
repeat.head(6)[["wo_id", "asset_id", "status", "date_closed", "last_modified"]]

Each pair is the same job exported twice: once while it was **Open**, again after it was **Closed**.
The `last_modified` timestamp tells us which version is newer.

**Rule:** for each `wo_id`, keep the version with the latest `last_modified`.

This is a defensible, written-down rule. "Keep the first one" is not; it depends on the order rows happened to be
in the file.

### Exercise 4.3 [Level 2: finish the code]
Sort by `last_modified`, then drop duplicates on `wo_id` keeping the **last** (newest) row.

In [ ]:
# TODO: sort wo by last_modified, then drop_duplicates with subset="wo_id" and keep="last"
wo_latest = wo   # replace this line

print(len(wo), "->", len(wo_latest))
print("wo_id unique?", wo_latest["wo_id"].is_unique)

In [ ]:
kept = wo_latest.set_index("wo_id").loc[repeat["wo_id"].unique(), "status"]
check(wo_latest["wo_id"].is_unique and (kept == "Closed").all(),
      "One row per work order, and every repeated job kept its newer Closed version.",
      "Either wo_id is not unique, or you kept the older (Open) version. Check keep='last' after sorting.")

In [ ]:
wo = wo_latest.sort_values("date_raised").reset_index(drop=True)

## 4. Conflicts you should not resolve automatically

### Exercise 4.4 [Level 2: finish the code]
The asset register should have one row per asset.

1. Drop exact duplicate rows.
2. Find any `asset_id` that still appears more than once. Those rows **disagree** with each other.
3. Save them to `outputs/asset_conflicts_for_review.csv` for the asset owner to resolve.
4. For now keep the first row, and add a `needs_review` column (True for conflicted assets) so the issue
   follows the data around.

In [ ]:
reg = reg.drop_duplicates()

# TODO: conflicts = rows where asset_id is duplicated (keep=False)
conflicts = reg.iloc[0:0]   # placeholder: an empty table

print("Conflicting rows:")
display(conflicts)

# TODO: save conflicts to outputs/asset_conflicts_for_review.csv

# TODO: add needs_review (True if asset_id is in conflicts), then keep only the first row per asset_id
reg["needs_review"] = False

print("Assets:", len(reg), " unique IDs:", reg["asset_id"].nunique())

In [ ]:
check(reg["asset_id"].is_unique and len(reg) == 60 and reg["needs_review"].sum() == 1,
      "60 unique assets, one flagged for review.",
      "Expected 60 unique asset IDs with exactly one needs_review = True.")

Look at the conflict: same asset, two different sites. Was it moved? Is one row a typo? The data cannot tell you.
A person can. Automating this decision would hide a real question about where that machine is.

## 5. Fuzzy matching: technician names

Names are a classic "same thing, different text" problem.

In [ ]:
wo["technician"].value_counts()

Eight technicians, around 30 spellings: case, missing fadas (Ciarán / Ciaran), initials, "Surname, First name".
Two steps:

1. **Normalise**: remove fadas and punctuation, flip "Murphy, Declan", lower case, single spaces.
2. **Match** what is left against the official staff list using a similarity score. Python's built-in `difflib`
   does this without installing anything.

### Exercise 4.5 [Level 1: fill the blanks]
The normalising function is written. Apply it and see how many spellings remain.

In [ ]:
import unicodedata

def normalise_name(name):
    """Lower-case, no fadas, no punctuation, 'Surname, First' flipped, single spaces."""
    if pd.isna(name):
        return name
    text = unicodedata.normalize("NFKD", str(name)).encode("ascii", "ignore").decode()   # á -> a
    text = text.replace("'", "").replace(".", " ")
    if "," in text:
        last, first = [part.strip() for part in text.split(",", 1)]
        text = f"{first} {last}"
    return " ".join(text.lower().split())

wo["tech_norm"] = wo["technician"].____(normalise_name)
print("Spellings before:", wo["technician"].nunique(), " after normalising:", wo["tech_norm"].____())

### Exercise 4.6 [Level 2: finish the code]
Match each normalised spelling to the staff list with `difflib.get_close_matches`. It returns a list of the closest
matches above a similarity `cutoff` (0 to 1). A high cutoff (0.85) means "only match if very similar".

In [ ]:
import difflib

STAFF = ["Aoife Byrne", "Ciarán Walsh", "Declan Murphy", "Niamh O'Connor",
         "Seán Kelly", "Orla Doyle", "Patrick Ryan", "Gráinne Fitzgerald"]
staff_by_norm = {normalise_name(s): s for s in STAFF}   # "ciaran walsh" -> "Ciarán Walsh"

print(difflib.get_close_matches("niamh o connor", staff_by_norm.keys(), n=1, cutoff=0.85))
print(difflib.get_close_matches("pat ryan", staff_by_norm.keys(), n=1, cutoff=0.85))

In [ ]:
auto_map = {}
for spelling in wo["tech_norm"].dropna().unique():
    matches = difflib.get_close_matches(spelling, staff_by_norm.keys(), n=1, cutoff=0.85)
    # TODO: if matches is not empty, set auto_map[spelling] to the official name: staff_by_norm[matches[0]]


unmatched = sorted(set(wo["tech_norm"].dropna()) - set(auto_map))
print("Matched automatically:", len(auto_map))
print("Need a human:", unmatched)

Initials (`d murphy`) and nicknames (`pat ryan`) do not score high enough. That is correct behaviour: if Kilbrack
hired a Deirdre Murphy next year, `d murphy` would be ambiguous. A person who knows the team confirms these.

### Exercise 4.7 [Level 2: finish the code]
Fill `MANUAL_MAP` for the names listed above (you are the person who knows the team), combine both maps and
apply. Then save the full mapping so someone can review it.

In [ ]:
MANUAL_MAP = {
    "pat ryan": "Patrick Ryan",
    # TODO: add the rest of the unmatched spellings
}
name_map = {**auto_map, **MANUAL_MAP}

wo["technician_raw"] = wo["technician"]
wo["technician"] = wo["tech_norm"].map(name_map)
print("Unmapped:", wo["technician"].isna().sum())
wo["technician"].value_counts()

In [ ]:
check(wo["technician"].isna().sum() == 0 and set(wo["technician"]) == set(STAFF),
      "Every work order now has one of the 8 official names.",
      "Some names are still unmapped. Look at the 'Need a human' list again.")

In [ ]:
tech_map = (wo[["technician_raw", "tech_norm", "technician"]].drop_duplicates()
              .assign(method=lambda d: np.where(d["tech_norm"].isin(MANUAL_MAP.keys()), "manual", "fuzzy"))
              .sort_values("technician"))
tech_map.to_csv("outputs/technician_map.csv", index=False)
tech_map.head(10)

## 6. Keys across systems: matching legacy IDs

Before the CMMS, Kilbrack used a spreadsheet. Its equipment IDs look like `CNC001` instead of `CNC-001`.
If we want to combine the two systems (next module) the keys have to match.

### Exercise 4.8 [Level 2: finish the code]
1. Measure the match rate against the register **before** fixing anything.
2. Write `normalise_asset_id(series)`: strip, upper, then insert the hyphen with a regular expression.
3. Measure again.

The regular expression is given: `r"^([A-Z]+)-?(\d+)$"` means "letters, an optional hyphen, digits".
`r"\1-\2"` rebuilds it as letters, hyphen, digits.

In [ ]:
legacy = pd.read_excel("data/work_orders_legacy.xlsx")
decom = pd.read_excel("data/asset_register.xlsx", sheet_name="Decommissioned")
known_ids = set(reg["asset_id"]) | set(decom["Asset ID"])   # in service + decommissioned

before = legacy["Equipment"].isin(known_ids).mean()
print(f"Legacy rows matching a known asset BEFORE: {before:.0%}")
legacy["Equipment"].sample(8, random_state=1).tolist()

In [ ]:
def normalise_asset_id(series):
    """'cnc001', ' CNC-001 ' and 'CNC001' all become 'CNC-001'."""
    s = series.astype(str).str.strip().str.upper()
    # TODO: apply s.str.replace(r"^([A-Z]+)-?(\d+)$", r"\1-\2", regex=True)
    return s

legacy["asset_id"] = normalise_asset_id(legacy["Equipment"])
after = legacy["asset_id"].isin(known_ids).mean()
print(f"Legacy rows matching a known asset AFTER: {after:.0%}")

In [ ]:
check(after == 1.0, "Every legacy job now links to a known asset (in service or decommissioned).",
      "Some legacy IDs still do not match. Print legacy.loc[~legacy['asset_id'].isin(known_ids), 'asset_id'].unique()")

Matching IDs is the simplest kind of **record linkage**. When there is no shared ID at all (the same job recorded in
two systems with different job numbers) you link on a combination of fields instead. That is Module 05.

## 7. Save outputs

In [ ]:
wo = wo.drop(columns=["tech_norm"])
wo.to_csv("outputs/work_orders_dedup.csv", index=False)
reg.to_csv("outputs/asset_register_dedup.csv", index=False)
print(wo.shape, reg.shape)

### Stretch [Level 3: Gemini allowed]
Ask Gemini: *"What are the risks of using fuzzy matching with a cutoff of 0.6 on technician names?"* Then try
`cutoff=0.6` and look at what gets matched. Find at least one match you would not accept.

## Summary

* Look at duplicates before deleting them, and write down why you removed them.
* Exact duplicates: `drop_duplicates()`. Duplicate keys: sort by a timestamp, keep the latest.
* When rows disagree and the data cannot say who is right, **flag and send to a human**.
* Fuzzy matching proposes; a person approves. Save the mapping so it can be audited and reused.
* Normalise keys (case, spaces, format) before you join anything.

Next: **05 Combining Data from Multiple Sources**.